# 13 · Does the planned approach also rescue all 10 demos?

Notebook 12 showed, on the 5 demos with similar grasp sides, that planning the approach to the bowl instead of copying mine raises the policy from 67% to 99%. Open question: do my 10 training demos, whose grasp sides differ by up to 60° (86-149° around the bowl), also reach that level once the approach is planned? Or does that inconsistency (H2) cost success on top?

**Experiment.** Regenerate notebook 02's data exactly (10 training demos, the same 600 seeds: same scenes, same grasp and place perturbations) with the approach planned straight down (`object_relative_vertical`), cut to 438 successes spread evenly over the demos (the size of `ours_all`), train and evaluate as every other run. Baseline: `ours_all` (my approach, 438 episodes) 57% [47-66].

In physics-only checks the teacher with the planned approach succeeds in 91% of the evaluation scenes for these 10 demos (83% with my approach).

**Predictions, fixed before running** (pooled over 2 evaluations, n = 100):
* >= 90%: grasp-side inconsistency does not matter once the approach is planned; the whole gap to teleop was the approach.
* <= 75%: inconsistent grasp sides cost success in addition (H2 matters on its own).
* In between: a small extra cost of inconsistency.

**Runtime:** T4, about 3 h.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"], check=True)
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO, SEED, STEPS, BATCH = "phone2panda-data", 0, 20000, 32
PER_DEMO, N_EPISODES = 60, 438                       # notebook 02's seeds (seed0 = 1000) and ours_all's size
NAME = "ours_all_vertical"
RUN = f"act_{NAME}_s{SEED}"
GEN = f"{BASE}/gen_all_vertical"
WORKERS = os.cpu_count()
from p2p.sim.env import render_speed
r = render_speed(); print(r)
assert r["gpu_rendering"], "rendering is on the CPU: Runtime -> Change runtime type -> T4 GPU, then restart"

## Generate: the 10 training demos (held-out demos excluded by default), resumable

In [ ]:
!python -m p2p.data.generate data/processed {GEN} --strategy object_relative_vertical --per-demo {PER_DEMO} --workers {WORKERS}

In [ ]:
import pandas as pd
log = pd.read_json(f"{GEN}/generation_log.jsonl", lines=True).drop_duplicates("seed", keep="last")
print("yield", round(log.success.mean(), 3), "successes", int(log.success.sum()), "of", len(log))
display(log.groupby("demo").success.agg(["mean", "sum"]))
assert log.success.sum() >= N_EPISODES

In [ ]:
!python -m p2p.data.lerobot_io ours {GEN} {BASE}/lerobot/{NAME} --balanced {N_EPISODES}
!python tools/demo_consistency.py {BASE}/lerobot/{NAME}
from p2p.hub import push
push(f"{BASE}/lerobot/{NAME}", HUB_REPO, path_in_repo=f"lerobot/{NAME}")

## Train and evaluate

In [ ]:
!python -m p2p.policy.act train {BASE}/lerobot/{NAME} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED}

In [ ]:
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 6 --out {BASE}/runs/{RUN}/eval
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --out {BASE}/runs/{RUN}/eval_rerun
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {BASE}/runs/{RUN}/eval_te
import glob
evals = " ".join(sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")))
!python tools/policy_failures.py {evals}
for e in sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")):
    print(e.split("/")[-2], json.dumps(json.load(open(e))["behaviour"]))

In [ ]:
from p2p.hub import push
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}")
!cp {GEN}/generation_log.jsonl {BASE}/runs/{RUN}/generation_log.jsonl
!cd {BASE}/runs && zip -qr {BASE}/{RUN}.zip {RUN} -x "*/final/*" "*/step_*"
try:
    from google.colab import files; files.download(f"{BASE}/{RUN}.zip")
except Exception:
    print("download", f"{BASE}/{RUN}.zip", "from the Output panel")